<a href="https://colab.research.google.com/github/ikabrain/UCS772-CV-NLP-Lab/blob/main/NLP_assign6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 6 - Text prediction using advanced method
---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

SEED = 42

## Preprocessing Text
---

### Initialising Dataset

In [2]:
import kagglehub
path = kagglehub.dataset_download("ashishpandey2062/next-word-predictor-text-generator-dataset")

100%|██████████| 61.5k/61.5k [00:00<00:00, 2.31MB/s]

Extracting files...


In [3]:
!ls {path}

next_word_predictor.txt


In [4]:
with open(path + "/next_word_predictor.txt", 'r', encoding='utf-8') as file:
    data = file.read()

print(data[:500])

The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. People were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. Children were playing games, and laughter filled the air.

As the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. Families gathered for picnics, and the smell of barbecues wafted through the air.


In [5]:
lines = [l.strip() for l in data.split("\n") if l.strip()]
print(lines[:3])

['The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. People were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. Children were playing games, and laughter filled the air.', 'As the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. Families gathered for picnics, and the smell of barbecues wafted through the air. It was a perfect day for a picnic by the lake.', 'In the distance, you could hear the sound of live music coming from a local band, and people began to gather around the stage to enjoy the performance. The atmosphere was electric, and the music had everyone swaying to the beat.']


### Exploring Corpus

In [6]:
import re
from collections import Counter

In [7]:
# Overall size
print("chars:", len(data))
print("lines:", len(lines), "| empty:", sum(1 for l in lines if not l.strip()))
print("words:", len(data.split()))

chars: 167445
lines: 1547 | empty: 0
words: 27644


In [8]:
# Most frequent words
words = re.findall(r"[a-z']+", data.lower())
print(Counter(words).most_common(50))

[('the', 1418), ('and', 859), ('a', 718), ('of', 689), ('to', 596), ('i', 431), ('you', 380), ('in', 307), ('is', 272), ('monica', 267), ('it', 248), ('with', 246), ('ross', 245), ('that', 238), ('rachel', 202), ('for', 187), ('chandler', 180), ('this', 166), ('on', 159), ('joey', 151), ('was', 144), ('oh', 136), ('phoebe', 135), ('are', 129), ('all', 123), ('as', 120), ('what', 119), ('be', 111), ('like', 111), ('no', 108), ("it's", 106), ("i'm", 103), ('her', 100), ('they', 98), ('just', 98), ('from', 95), ('okay', 94), ('not', 93), ('so', 89), ('my', 89), ('have', 87), ('me', 86), ('where', 83), ('she', 81), ('know', 80), ('we', 79), ('out', 77), ('well', 77), ('their', 77), ('can', 77)]


In [9]:
# Shortest non-empty lines
nonempty = [l.strip() for l in lines if l.strip()] # Redundant as no empty line is read in this case!
for l in sorted(set(nonempty), key=len)[:40]:
    print(repr(l))

'End'
'Usage'
'(Pause)'
'Kid: No.'
'Encoding'
'Ross:Hi.'
'Joey: So?'
'All: Yes!'
'All: Mmm.'
'All: Eww!'
'Ross: Hi.'
'Alan: Hi.'
'Ross: Mm.'
'Ross: Why?'
'Joey: Ohh.'
'Alan: Wow.'
'Joey: Hey!'
'All: Nooo!'
'Susan: Hi.'
'All: Yeah!'
'Carol: So.'
'Paula: No.'
'All: What?'
'(Silence.)'
'Ross: Easy.'
'Characters:'
'Ross: Ohhh.'
'Susan: She.'
'Ross: Yeah.'
'Joey: What?'
'Phoebe: No.'
'Ross: What?'
'Joey: Okay!'
'[SCENE END]'
'Paul: Yeah?'
'Monica: No.'
'Joey: Alan.'
'Monica: Yes!'
'Phoebe: Huh!'
'All: Eeaagh!'


We see that there's a script of the TV show "Friends" here with speaker labels, that can give us some trouble. Let's handle that.

In [10]:
# Catch speaker labels
prefixes = Counter(m.group(1) for l in nonempty
                   if (m := re.match(r"^([A-Z][\w .,'-]{0,30}):", l)))
print(prefixes.most_common(30))

[('Monica', 200), ('Ross', 183), ('Rachel', 155), ('Chandler', 123), ('Phoebe', 102), ('Joey', 99), ('All', 35), ('Headline', 24), ('Carol', 21), ('Paul', 17), ('Barry', 16), ('Lizzie', 11), ('Sarah', 10), ('Mrs. Geller', 10), ('Mark', 9), ('Susan', 9), ('Pizza Guy', 7), ('Paula', 6), ('Alan', 6), ('Receptionist', 6), ('Frannie', 5), ('Mr. Geller', 5), ('Robbie', 5), ('Joanne', 5), ('Written by', 4), ('Marsha', 4), ('Kiki', 4), ('Main article', 3), ('Leslie', 3), ('Kid', 3)]


In [11]:
# Bracketed content
brackets = re.findall(r"\([^)]*\)|\[[^\]]*\]", data)
print("Parenthesized content: ", len(brackets))
print(brackets[:5], "..." if len(brackets) > 5 else "")
print()

# Unbalanced parentheses within a line
bad = [l for l in nonempty if l.count("(") != l.count(")")]
print("No. of lines with unbalanced parethesis: ", len(bad))
print(bad[:5], "..." if len(bad) > 5 else "")

Parenthesized content:  449
['(AI)', '(BCI)', '(10,000 words continued)', '(IoT)', '[SCENE START]'] ...

No. of lines with unbalanced parethesis:  1
['Rachel: Oh my God! (Rachel, Leslie, Kiki, and Joanne all scream and hug each other.'] 


In [12]:
# Non-ASCII characters
non_ascii = Counter(c for c in data if ord(c) > 127)
print(non_ascii.most_common())

[('—', 11), ('í', 5), ('á', 2), ('…', 2), ('ö', 1), ('¬', 1), ('–', 1)]


In [13]:
# Punctuation characters
punct = Counter(c for c in data if not c.isalnum() and not c.isspace())
print(punct.most_common())

[('.', 2513), (',', 2233), (':', 1198), ("'", 1007), ('?', 431), ('-', 384), ('(', 379), (')', 378), ('!', 377), ('"', 88), ('[', 73), (']', 73), (';', 12), ('—', 11), ('/', 9), ('+', 4), ('&', 3), ('…', 2), ('%', 1), ('$', 1), ('¬', 1), ('`', 1), ('=', 1), ('–', 1)]


In [14]:
# Duplicate lines
dupes = [l for l, c in Counter(nonempty).items() if c > 1]
print("Exact duplicate lines:", len(dupes))
print(dupes[:5], "..." if len(dupes) > 5 else "")

# Near-duplicate lines with same starts
starts = Counter(l[:40].lower() for l in nonempty if len(l) > 40)
print([s for s, c in starts.items() if c > 1])

Exact duplicate lines: 13
['Note: This is a sample of the 10,000-word dataset. The full dataset will be generated upon confirmation.', 'Characters:', '[SCENE START]', 'Written by: Marta Kauffman & David Crane', 'Commercial Break'] ...
['explorers ventured deep into the amazon,', 'note: this is a sample of the 10,000-wor', "[scene: monica and rachel's, everyone is", '[scene: madison square garden, the guys ', "[scene: monica and rachel's, the girls a"]


In [15]:
# Looking for meta-text from LLM yapping :/
meta = re.compile(r"rnn model|dataset|certainly|please let me know|note:|sample", re.I)
hits = [l for l in nonempty if meta.search(l)]
print("Found meta-text:", len(hits))
print()
for l in hits:
    print("-", l[:100])

Found meta-text: 17

- Your RNN model can use this text to predict the next word in a sequence, offering an exciting opport
- The Amazon was a place of awe and wonder, teeming with life and secrets waiting to be uncovered. You
- The Amazon was a realm of magic and mystery, a treasure trove of life and enigma waiting to be unrav
- The Faroe Islands were a place of rugged beauty, where human resilience and the forces of nature int
- Machu Picchu was a sanctuary of history, a place where the past and present intertwined. Your RNN mo
- Iceland was a realm of elemental grandeur, a place where the forces of nature shaped every aspect of
- The Sahara was a realm of solitude and wonder, where humans and nature coexisted in a delicate balan
- Bhutan was a realm of spiritual harmony and environmental stewardship, where ancient traditions and 
- Certainly, here's another unique and extended text passage for your next word predictor project:
- Madagascar was a realm of evolution's marvels and cult

### Text Cleaning

In [16]:
DROP_STARTS = ("certainly", "note:", "please let me know", "this is just", "this sample", "tech news dataset", "written by", "remember, the key")

In [17]:
import re, unicodedata

def clean_line(line):
    line = re.sub(r"Your RNN model[^.]*\.", " ", line)       # meta sentence at the end of paragraphs
    line = re.sub(r"\(.*?\)|\[.*?\]", " ", line)             # stage directions, [Scene: ...], [1], etc etc
    line = re.sub(r"\(.*$", " ", line)                       # unclosed "(" runs to end of line
    line = line.replace(")", " ")                            # stray ")"
    line = re.sub(r"^[A-Z][\w .,'-]{0,30}:\s*", "", line)    # "Monica: ", "Headline: ", "Setting: ", other speaker calls
    line = line.replace("\u2019", "'").replace("\u2018", "'")
    line = re.sub(r"\u2014|\u2013|\u2026|--", " ", line)     # handling dashes and the ellipsis character
    line = unicodedata.normalize("NFKD", line).encode("ascii", "ignore").decode()  # García -> Garcia
    return re.sub(r"\s+", " ", line).strip()

In [18]:
def clean_text(data):
    out = []
    for line in data.split("\n"):
        if line.strip().lower().startswith(DROP_STARTS):
            continue
        line = clean_line(line)
        if len(line.split()) >= 3:      # also drops "End", "Commercial Break" and empty lines
            out.append(line)
    return out

In [19]:
# Final cleaned text
cleaned = clean_text(data)
print("No. of cleaned lines: ", len(cleaned))
print("- " + "\n- ".join(cleaned[:10]))

No. of cleaned lines:  1164
- The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. People were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. Children were playing games, and laughter filled the air.
- As the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. Families gathered for picnics, and the smell of barbecues wafted through the air. It was a perfect day for a picnic by the lake.
- In the distance, you could hear the sound of live music coming from a local band, and people began to gather around the stage to enjoy the performance. The atmosphere was electric, and the music had everyone swaying to the beat.
- As the stars began to twinkle in the night sky, the crowd grew even larger, and the festivities continued well into the night. It was a day filled with joy, laughter, and memories that wou

In [20]:
lines = list(dict.fromkeys(cleaned)) # Drop exact duplicate lines, BUT keep order!

### Train-Val-Test Sentence Split

In [21]:
rng = np.random.default_rng(seed=SEED)
rng.shuffle(lines)

In [22]:
n = len(lines)
i1, i2 = int(0.8 * n), int(0.9 * n) # Using 80-10-10 split
train_lines, val_lines, test_lines = lines[:i1], lines[i1:i2], lines[i2:]

for name, part in {"training": train_lines, "validation": val_lines, "testing": test_lines}.items():
    print(name.capitalize()+":", len(part), "lines,", sum(len(l.split()) for l in part), "words")

Training: 928 lines, 18558 words
Validation: 116 lines, 2319 words
Testing: 117 lines, 1843 words


In [23]:
# Sentence-splitting
def to_sentences(lines):
    out = []
    for line in lines:
        line = re.sub(r"\b(Mr|Mrs|Ms|Dr|St)\.", r"\1", line)   # don't split on titles
        for s in re.split(r"(?<=[.!?])\s+", line):
            if len(s.split()) >= 2:
                out.append(s)
    return out

train_sents = to_sentences(train_lines)
val_sents   = to_sentences(val_lines)
test_sents  = to_sentences(test_lines)
print("Training:", len(train_sents), "sentences")
print("Validation:", len(val_sents), "sentences")
print("Testing:", len(test_sents), "sentences")

Training: 1698 sentences
Validation: 208 sentences
Testing: 180 sentences


In [24]:
# Checking sentence lengths for max sequence length
lens = np.array([len(s.split()) for s in train_sents])
print("Training has", len(lens), "sentences.")
print("Length percentiles (50/90/95/99/max):", np.percentile(lens, [50, 90, 95, 99]), lens.max())

Training has 1698 sentences.
Length percentiles (50/90/95/99/max): [ 8.   22.   26.   34.03] 53


### Tokenization

In [25]:
# Size of vocabulary across training sents for max tokens
counts = Counter(w for s in train_sents for w in re.findall(r"[a-z0-9']+", s.lower()))
print("Training has", len(counts), "unique words.")
print("Words seen at least twice:", sum(1 for c in counts.values() if c >= 2))

Training has 4022 unique words.
Words seen at least twice: 1794


In [ ]:
MAX_WORDS = 2000
SEQ_LEN = 30

In [ ]:
def standardize(x):
    x = tf.strings.lower(x)
    x = tf.strings.regex_replace(x, r"[^a-z0-9\s']", " ")
    x = tf.strings.regex_replace(x, r"(^|\s)'+|'+(\s|$)", " ")
    x = tf.strings.strip(tf.strings.regex_replace(x, r"\s+", " "))
    return tf.strings.join(["xxbos", x], separator=" ")

In [ ]:
vectorizer = layers.TextVectorization(
    max_tokens=MAX_WORDS+3,
    standardize=standardize,
    output_mode="int",
    output_sequence_length=SEQ_LEN+1, # xxbos + 30 words; makes sure to pre-pad and pre-truncate
)

vectorizer.adapt(train_sents)

vocab = vectorizer.get_vocabulary()

[np.str_('achieve'),
 np.str_('accuracy'),
 np.str_('accountable'),
 np.str_('accountability'),
 np.str_('accompanied'),
 np.str_('accommodate'),
 np.str_('accidentally'),
 np.str_('accessible'),
 np.str_('accented'),
 np.str_('acacia'),
 np.str_('abundant'),
 np.str_('absolutely'),
 np.str_('ability'),
 np.str_('abandoned'),
 np.str_('abandon'),
 np.str_('aaand'),
 np.str_('aa'),
 np.str_('8th'),
 np.str_('88591'),
 np.str_('850'),
 np.str_('8'),
 np.str_('7'),
 np.str_('64'),
 np.str_('630'),
 np.str_('6'),
 np.str_('53qubit'),
 np.str_('40s'),
 np.str_('4'),
 np.str_('31'),
 np.str_('30s'),
 np.str_('300'),
 np.str_('26'),
 np.str_('255'),
 np.str_('20s'),
 np.str_('2022'),
 np.str_('200'),
 np.str_('20'),
 np.str_('2'),
 np.str_('19th'),
 np.str_('1960s'),
 np.str_('180'),
 np.str_('159'),
 np.str_('127'),
 np.str_('1114112'),
 np.str_('100000'),
 np.str_('10000'),
 np.str_('100'),
 np.str_('0–31'),
 np.str_('09'),
 np.str_('0')]

## Q1: Apply deep learning models i.e. LSTM for next word prediction for sentiment analysis data
---

## Q2: Comment based on the various evaluation metrics for your results.
---

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay